# Lesson 07：CTE 與 SQL Window Functions

示範如何在 SQLite 中使用 CTE 與 window functions 做時間序列、排名與前後期比較分析。

學習目標：
- 使用 CTE 先整理中間結果
- 計算累積營收與 7 日移動平均
- 比較 `ROW_NUMBER`、`RANK`、`DENSE_RANK`
- 用 `LAG` 與 `LEAD` 做日變化與月成長率分析


## 1. 載入套件與建立連線

本課沿用 `data/raw/course.db`。`pd.read_sql_query()` 可以把 SQL 查詢結果直接轉成 pandas DataFrame，方便在 notebook 中觀察。


In [1]:
import sqlite3
import pandas as pd
from common import RAW, ensure_packages

ensure_packages()

db_path = RAW / "course.db"
conn = sqlite3.connect(db_path)

print(db_path)


E:\py_20260620\data\raw\course.db


## 2. CTE：先建立每張訂單的營收

CTE 是 Common Table Expression，可以把一段查詢命名，讓後面的 SQL 更容易閱讀。這裡先把 `order_items` 彙總成 `order_rev`，每列代表一張訂單的營收。

語法重點：
- `WITH order_rev AS (...)` 建立暫時查詢結果
- 後續 SQL 可以像使用資料表一樣使用 `order_rev`


In [2]:
order_rev_query = """
WITH order_rev AS (
    SELECT
        order_id,
        SUM(quantity * unit_price * (1 - discount_rate)) AS revenue
    FROM order_items
    GROUP BY order_id
)
SELECT *
FROM order_rev
LIMIT 5;
"""

pd.read_sql_query(order_rev_query, conn)


,order_id,revenue
0,1,538.65
1,2,4357.70
2,3,3536.00
3,4,1193.40
4,5,13175.50


## 3. CTE 改寫：依付款方式計算平均營收

先用 CTE 算每張訂單營收，再與 `orders` 合併，最後依 `payment_type` 彙總。


In [3]:
payment_cte_query = """
WITH order_rev AS (
    SELECT
        order_id,
        SUM(quantity * unit_price * (1 - discount_rate)) AS revenue
    FROM order_items
    GROUP BY order_id
)
SELECT
    o.payment_type,
    COUNT(*) AS orders,
    ROUND(AVG(r.revenue), 2) AS avg_revenue
FROM order_rev r
JOIN orders o ON r.order_id = o.order_id
WHERE o.status = 'completed'
GROUP BY o.payment_type
ORDER BY avg_revenue DESC;
"""

payment_cte_result = pd.read_sql_query(payment_cte_query, conn)
payment_cte_result


,payment_type,orders,avg_revenue
0,cod,2060,5052.66
1,wallet,3076,4987.98
2,atm,5111,4979.96
3,card,10166,4974.20


## 4. Window Function：累積營收與 7 日移動平均

Window function 會在「保留原本列」的情況下，計算一段視窗範圍內的值。這裡先建立每日營收 `daily`，再計算：

- `running_rev`：從第一天累積到當天的營收
- `ma7`：當天與前 6 天的 7 日移動平均


In [5]:
daily_window_query = """

    SELECT
        substr(o.order_date, 1, 10) AS d,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS rev
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY substr(o.order_date, 1, 10)

"""
pd.read_sql_query(daily_window_query, conn)

,d,rev
0,2024-01-01,172521.55
1,2024-01-02,125992.90
2,2024-01-03,190549.30
3,2024-01-04,176488.30
4,2024-01-05,86529.65
...,...,...
726,2025-12-27,129858.30
727,2025-12-28,138155.15
728,2025-12-29,171045.65
729,2025-12-30,158274.35


In [9]:
daily_window_query = """
WITH daily AS (--CTE語法, 建立臨時資料表--
    SELECT
        substr(o.order_date, 1, 10) AS d, --substr()取前10個字元, 也就是西元年月日--
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS rev
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY substr(o.order_date, 1, 10)
)
SELECT
    d,
    ROUND(rev, 2) AS rev,
    ROUND(
        SUM(rev) OVER (--OVER代表不要縮成一列--
            ORDER BY d
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW --從第一筆開始累加--
        ),
        2
    ) AS running_rev,
    ROUND(
        AVG(rev) OVER (
            ORDER BY d
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW --當筆+前6筆=7日平均--
        ),
        2
    ) AS ma7
FROM daily
ORDER BY d
LIMIT 10;
"""

daily_window = pd.read_sql_query(daily_window_query, conn)
daily_window


,d,rev,running_rev,ma7
0,2024-01-01,172521.55,172521.55,172521.55
1,2024-01-02,125992.90,298514.45,149257.22
2,2024-01-03,190549.30,489063.75,163021.25
3,2024-01-04,176488.30,665552.05,166388.01
4,2024-01-05,86529.65,752081.70,150416.34
5,2024-01-06,153786.85,905868.55,150978.09
6,2024-01-07,128666.20,1034534.75,147790.68
7,2024-01-08,130716.90,1165251.65,141818.59
8,2024-01-09,176540.35,1341792.00,149039.65
9,2024-01-10,157974.05,1499766.05,144386.04


## 5. 顧客營收排名

`RANK() OVER (ORDER BY total_rev DESC)` 會依照顧客總消費由高到低排名。若有相同金額，`RANK()` 會給相同名次，下一個名次會跳號。


In [10]:
customer_rank_query = """
WITH customer_rev AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS total_rev
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id
)
SELECT
    customer_id,
    ROUND(total_rev, 2) AS total_rev,
    RANK() OVER (ORDER BY total_rev DESC) AS revenue_rank
FROM customer_rev
LIMIT 20;
"""

customer_rank = pd.read_sql_query(customer_rank_query, conn)
customer_rank


,customer_id,total_rev,revenue_rank
0,541,106597.85,1
1,1717,100774.50,2
2,2079,100759.10,3
3,1483,100045.75,4
4,53,100029.40,5
5,1729,99322.95,6
6,18,98810.60,7
7,1602,98540.70,8
8,1705,96237.65,9
9,2275,95695.25,10


## 6. 比較 ROW_NUMBER、RANK、DENSE_RANK

三個排名函數很像，但處理同分時不同：

- `ROW_NUMBER()`：每列都有唯一序號，不管是否同分
- `RANK()`：同分同名次，下一名會跳號
- `DENSE_RANK()`：同分同名次，下一名不跳號


In [13]:
rank_compare_query = """
WITH customer_rev AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS total_rev
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id
)
SELECT
    customer_id,
    ROUND(total_rev, 2) AS total_rev,
    ROW_NUMBER() OVER (ORDER BY total_rev DESC) AS row_num,
    RANK()       OVER (ORDER BY total_rev DESC) AS rnk,
    DENSE_RANK() OVER (ORDER BY total_rev DESC) AS dense_rnk
FROM customer_rev
LIMIT 10;
"""

rank_compare = pd.read_sql_query(rank_compare_query, conn)
rank_compare


,customer_id,total_rev,row_num,rnk,dense_rnk
0,541,106597.85,1,1,1
1,1717,100774.50,2,2,2
2,2079,100759.10,3,3,3
3,1483,100045.75,4,4,4
4,53,100029.40,5,5,5
5,1729,99322.95,6,6,6
6,18,98810.60,7,7,7
7,1602,98540.70,8,8,8
8,1705,96237.65,9,9,9
9,2275,95695.25,10,10,10


## 7. LAG 與 LEAD：比較前一天與後一天

`LAG()` 可以拿到前一列的值，`LEAD()` 可以拿到後一列的值。對時間序列來說，這很適合做前後期比較，例如日營收變化。


In [14]:
daily_lag_lead_query = """
WITH daily AS (
    SELECT
        substr(o.order_date, 1, 10) AS d,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS rev
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY substr(o.order_date, 1, 10)
)
SELECT
    d,--日期--
    ROUND(rev, 2) AS rev,--當天營收--
    ROUND(LAG(rev, 1) OVER (ORDER BY d), 2) AS prev_day_rev,--前一天營收--
    ROUND(LEAD(rev, 1) OVER (ORDER BY d), 2) AS next_day_rev,--NEXT天營收--
    ROUND(rev - LAG(rev, 1) OVER (ORDER BY d), 2) AS day_over_day --差額--
FROM daily
ORDER BY d
LIMIT 10;
"""

daily_lag_lead = pd.read_sql_query(daily_lag_lead_query, conn)
daily_lag_lead


,d,rev,prev_day_rev,next_day_rev,day_over_day
0,2024-01-01,172521.55,NaN,125992.90,NaN
1,2024-01-02,125992.90,172521.55,190549.30,-46528.65
2,2024-01-03,190549.30,125992.90,176488.30,64556.40
3,2024-01-04,176488.30,190549.30,86529.65,-14061.00
4,2024-01-05,86529.65,176488.30,153786.85,-89958.65
5,2024-01-06,153786.85,86529.65,128666.20,67257.20
6,2024-01-07,128666.20,153786.85,130716.90,-25120.65
7,2024-01-08,130716.90,128666.20,176540.35,2050.70
8,2024-01-09,176540.35,130716.90,157974.05,45823.45
9,2024-01-10,157974.05,176540.35,144529.60,-18566.30


## 8. PARTITION BY：依付款方式計算月成長率

`PARTITION BY payment_type` 會讓每種付款方式各自形成一組視窗。也就是說，信用卡只和信用卡的前一個月比較，電子錢包只和電子錢包的前一個月比較。

`NULLIF(..., 0)` 可避免前月營收為 0 時發生除以零。


In [15]:
monthly_mom_query = """
WITH monthly AS (
    SELECT
        substr(o.order_date, 1, 7) AS ym,
        o.payment_type,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY substr(o.order_date, 1, 7), o.payment_type
)
SELECT
    ym,
    payment_type,
    ROUND(revenue, 2) AS revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (PARTITION BY payment_type ORDER BY ym))
        / NULLIF(LAG(revenue) OVER (PARTITION BY payment_type ORDER BY ym), 0),
        4
    ) AS mom_change
FROM monthly
ORDER BY payment_type, ym;
"""

monthly_mom = pd.read_sql_query(monthly_mom_query, conn)
monthly_mom.head(20)


,ym,payment_type,revenue,mom_change
0,2024-01,atm,1176323.10,NaN
1,2024-02,atm,1069209.15,-0.0911
2,2024-03,atm,1138525.25,0.0648
3,2024-04,atm,1015910.60,-0.1077
4,2024-05,atm,1050332.45,0.0339
5,2024-06,atm,1002494.55,-0.0455
6,2024-07,atm,1039476.20,0.0369
7,2024-08,atm,1006122.55,-0.0321
8,2024-09,atm,1091623.80,0.0850
9,2024-10,atm,1148221.95,0.0518


## 9. 小練習

請建立一個查詢，計算每個月的總營收，並加上：

- `prev_month_revenue`：前一個月營收
- `mom_change`：月成長率

提示：可以先用 CTE 建立 `monthly`，再用 `LAG(revenue) OVER (ORDER BY ym)` 取得前月營收。


In [ ]:
monthly_total_query = """
WITH monthly AS (
    SELECT
        substr(o.order_date, 1, 7) AS ym,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY substr(o.order_date, 1, 7)
)
SELECT
    ym,
    ROUND(revenue, 2) AS revenue,
    ROUND(LAG(revenue) OVER (ORDER BY ym), 2) AS prev_month_revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (ORDER BY ym))
        / NULLIF(LAG(revenue) OVER (ORDER BY ym), 0),
        4
    ) AS mom_change
FROM monthly
ORDER BY ym;
"""

monthly_total = pd.read_sql_query(monthly_total_query, conn)
monthly_total


## 10. 常見錯誤與延伸

常見錯誤：
- 忘記 `ORDER BY`，導致 window function 的前後順序不可靠。
- 在明細層級直接排名，結果排名的是商品列，不是訂單或顧客。
- 需要分組比較時忘記 `PARTITION BY`，讓不同類別混在同一個視窗裡。

延伸練習：
- 把每日營收的移動平均改成 14 日移動平均。
- 將顧客排名接上 `customers` 表，觀察高消費顧客的 `segment` 與 `city`。


## 11. 關閉資料庫連線

分析完成後關閉連線，釋放資料庫資源。


In [16]:
conn.close()
print("資料庫連線已關閉。")


資料庫連線已關閉。
